# enrichment pipeline flowchart with branch coverage
- for each decision point: how many EMC planet rows take each branch; sources NEA → EU → EPIC → TOI → SIMBAD
- the mermaid source is written next to this notebook and rendered with a local `mmdc`
- exports to `report/figures/enrichment_pipeline_flowchart.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import MaskedColumn

from crossmatching import EMCCatalog
from crossmatching.enrichment import rocky_mask, temperate_mask
from report_plots import common, data


In [ ]:
emc = EMCCatalog().load(from_file=str(data.EMC_CSV_PATH), format='csv')
merger = data.build_param_filler()

out = merger.enrich(
    emc,
    planet_flux_key="flux_rel",
    **EMCCatalog.ENRICH_KEYS,
)[0]
n = len(out)
n_hosts = len({str(v) for v in out['main_id']})
print(f'{n:,} planet rows  |  {n_hosts:,} unique hosts')


In [ ]:
from IPython.display import Markdown, display
import astropy.units as u

def _finite(col):
    if isinstance(col, MaskedColumn):
        return ~np.ma.getmaskarray(col)
    return np.isfinite(np.asarray(col, dtype=float))

teff_src = [str(v) for v in out['st_teff_src']]
rad_src  = [str(v) for v in out['st_rad_src']]
mass_src = [str(v) for v in out['st_mass_src']]
a_src    = [str(v) for v in out['a_src']]
flux_src = [str(v) for v in out['flux_rel_src']]
lum_src  = [str(v) for v in out['st_lum_src']]
logg_src_col = [str(v) for v in out['st_logg_src']]
spec_src_col = [str(v) for v in out['st_spectype_src']]
kmag_src_col = [str(v) for v in out['sy_kmag_src']]
met_src_col  = [str(v) for v in out['st_met_src']]
vmag_src_col = [str(v) for v in out['sy_vmag_src']]
dist_src_col = [str(v) for v in out['sy_dist_src']]
eqt_src_col  = [str(v) for v in out['pl_eqt_src']]
pl_mass_src_col = [str(v) for v in out['pl_mass_src']]
msini_src_col = [str(v) for v in out['msini_src']]
p_src_col    = [str(v) for v in out['p_src']]

r_direct = _finite(out['r'])
r_bounds = _finite(out['r_lower_bound'])
r_none   = ~r_direct & ~r_bounds
lum_ok   = _finite(out['st_lum'])

out['mass_earth'] = out['mass'] * u.M_jupiter.to(u.M_earth)
out['msini_earth'] = out['msini'] * u.M_jupiter.to(u.M_earth)
out['r_earth'] = out['r'] * u.R_jupiter.to(u.R_earth)
out['r_earth_err1'] = out['r_max'] * u.R_jupiter.to(u.R_earth)
out['r_earth_err2'] = out['r_min'] * u.R_jupiter.to(u.R_earth)

out['r_lower_bound_earth'] = out['r_lower_bound'] * u.R_jupiter.to(u.R_earth)
out['r_upper_bound_earth'] = out['r_upper_bound'] * u.R_jupiter.to(u.R_earth)
HZ_L, HZ_U = 0.35, 1.77
RK_L, RK_U = 0.5, 1.5
rocky_conf = rocky_mask(out['r_earth'], None, None, out['r_lower_bound_earth'], out['r_upper_bound_earth'],
                        lower=RK_L, upper=RK_U)
rocky_unc  = (rocky_mask(out['r_earth'], None, None, out['r_lower_bound_earth'], out['r_upper_bound_earth'],
                         lower=RK_L, upper=RK_U, use_interval=True) & ~rocky_conf)
temperate  = temperate_mask(out['flux_rel'], out['flux_rel_max'], out['flux_rel_min'],
                            lower=HZ_L, upper=HZ_U)

def _c(x):
    return str(x)

counts = {
    'n_total':        f'{n:,}',
    'n_teff_nea':     _c(sum(s == 'nea'             for s in teff_src)),
    'n_teff_epic':    _c(sum(s == 'epic'            for s in teff_src)),
    'n_teff_toi':     _c(sum(s == 'toi'             for s in teff_src)),
    'n_teff_simbad':  _c(sum(s == 'simbad'          for s in teff_src)),
    'n_teff_none':    _c(sum(s == ''                for s in teff_src)),
    'n_rad_nea':      _c(sum(s == 'nea'             for s in rad_src)),
    'n_rad_epic':     _c(sum(s == 'epic'            for s in rad_src)),
    'n_rad_toi':      _c(sum(s == 'toi'             for s in rad_src)),
    'n_rad_simbad':   _c(sum(s == 'simbad'          for s in rad_src)),
    'n_rad_zams':     _c(sum(s.startswith('ms(')    for s in rad_src)),
    'n_rad_direct':   _c(sum(s in ('nea','epic','toi') for s in rad_src)),
    'n_rad_calc':     _c(sum(s.startswith('ms(')    for s in rad_src)),
    'n_rad_none':     _c(sum(s == ''                for s in rad_src)),
    'n_mass_nea':     _c(sum(s == 'nea'             for s in mass_src)),
    'n_mass_epic':    _c(sum(s == 'epic'            for s in mass_src)),
    'n_lum':          _c(np.sum(lum_ok)),
    'n_lum_nea':      _c(sum(s == 'nea'         for s in lum_src)),
    'n_lum_epic':     _c(sum(s == 'epic'        for s in lum_src)),
    'n_lum_direct':   _c(sum(s in ('nea', 'epic') for s in lum_src)),
    'n_lum_calc':     _c(sum(s.startswith('derived(rad:') for s in lum_src)),
    'n_a_direct':     _c(sum(s == 'input'             for s in a_src)),
    'n_a_kepler':     _c(sum(s.startswith('kepler') for s in a_src)),
    'n_a_none':       _c(sum(s == ''                for s in a_src)),
    'n_p_valid':      _c(int(np.sum(_finite(out['p'])))),
    'n_flux_direct':  _c(sum(s in ('nea', 'epic', 'toi') for s in flux_src)),
    'n_flux_nea':     _c(sum(s == 'nea'       for s in flux_src)),
    'n_flux_epic':    _c(sum(s == 'epic'      for s in flux_src)),
    'n_flux_toi':     _c(sum(s == 'toi'       for s in flux_src)),
    'n_flux_comp':    _c(sum(s.startswith('derived(')      for s in flux_src)),
    'n_flux_none':    _c(sum(s == ''                for s in flux_src)),
    'n_r_direct':     _c(np.sum(r_direct)),
    'n_r_msini':      _c(np.sum(r_bounds)),
    'n_rocky_conf':   _c(np.sum(rocky_conf)),
    'n_rocky_unc':    _c(np.sum(rocky_unc)),
    'n_temperate':    _c(np.sum(temperate)),
    'n_logg_nea':     _c(sum(s == 'nea'            for s in logg_src_col)),
    'n_logg_epic':    _c(sum(s == 'epic'           for s in logg_src_col)),
    'n_logg_toi':     _c(sum(s == 'toi'            for s in logg_src_col)),
    'n_logg_simbad':  _c(sum(s == 'simbad'         for s in logg_src_col)),
    'n_mass_logg':    _c(sum(s.startswith('logg_derived') for s in mass_src)),
    'n_spec_nea':     _c(sum(s == 'nea'    for s in spec_src_col)),
    'n_spec_epic':    _c(sum(s == 'epic'   for s in spec_src_col)),
    'n_spec_simbad':  _c(sum(s == 'simbad' for s in spec_src_col)),
    'n_spec_zams':    _c(sum(s == ''       for s in spec_src_col)),
    'n_zams_with_spec': _c(sum(1 for rs, ss in zip(rad_src, spec_src_col)
                                if rs.startswith('ms(') and ss != '')),
    'n_kmag_nea':     _c(sum(s == 'nea'    for s in kmag_src_col)),
    'n_kmag_epic':    _c(sum(s == 'epic'   for s in kmag_src_col)),
    'n_kmag_simbad':  _c(sum(s == 'simbad' for s in kmag_src_col)),
    'n_met_nea':      _c(sum(s == 'nea'    for s in met_src_col)),
    'n_met_epic':     _c(sum(s == 'epic'   for s in met_src_col)),
    'n_met_toi':      _c(sum(s == 'toi'    for s in met_src_col)),
    'n_met_simbad':   _c(sum(s == 'simbad' for s in met_src_col)),
    'n_met_mann_mks': _c(sum(rs.startswith('mann_mks')  and ms != '' for rs, ms in zip(rad_src, met_src_col))),
    'n_met_torres':   _c(sum(rs.startswith('torres')    and ms != '' for rs, ms in zip(rad_src, met_src_col))),
    'n_met_mann_teff':_c(sum(rs.startswith('mann_teff') and ms != '' for rs, ms in zip(rad_src, met_src_col))),
    'n_rad_mann_mks': _c(sum(s.startswith('mann_mks')  for s in rad_src)),
    'n_rad_torres':   _c(sum(s.startswith('torres')    for s in rad_src)),
    'n_rad_mann_teff':_c(sum(s.startswith('mann_teff') for s in rad_src)),
    'n_teff_eu':      _c(sum(s == 'eu' for s in teff_src)),
    'n_rad_eu':       _c(sum(s == 'eu' for s in rad_src)),
    'n_mass_eu':      _c(sum(s == 'eu' for s in mass_src)),
    'n_met_eu':       _c(sum(s == 'eu' for s in met_src_col)),
    'n_spec_eu':      _c(sum(s == 'eu' for s in spec_src_col)),
    'n_kmag_eu':      _c(sum(s == 'eu' for s in kmag_src_col)),
}

counts["n_rad_calc"] = counts["n_rad_zams"]
counts["n_rad_direct"] = str(int(counts["n_rad_nea"]) \
                       + int(counts["n_rad_eu"]) \
                       + int(counts["n_rad_epic"]) \
                       + int(counts["n_rad_toi"]) \
                       + int(counts["n_rad_simbad"]))

branches = [
    (r'$T_{\mathrm{eff}}$ source', [
        ('NEA',          int(sum(s == "nea"    for s in teff_src))),
        ('EU',           int(sum(s == "eu"       for s in teff_src))),
        ('EPIC',         int(sum(s == "epic"   for s in teff_src))),
        ('TOI',          int(sum(s == "toi"    for s in teff_src))),
        ('SIMBAD',       int(sum(s == "simbad" for s in teff_src))),
        ('ZAMS fallback',int(sum(s.startswith("spectype_derived") or s.startswith("StephanBoltzmann_derived") for s in teff_src))),
        ('Unavailable',  int(sum(s == ""       for s in teff_src))),
    ], False),
    (r'$R_\star$ source', [
        ('NEA',                int(sum(s == "nea"             for s in rad_src))),
        ('EU',                 int(sum(s == "eu"                for s in rad_src))),
        ('EPIC',               int(sum(s == "epic"            for s in rad_src))),
        ('TOI',                int(sum(s == "toi"             for s in rad_src))),
        ('SIMBAD',             int(sum(s == "simbad"          for s in rad_src))),
        ("Mann 2015 ($M_{K_s}$)", int(sum(s.startswith("mann_mks")  for s in rad_src))),
        ('Torres 2010',        int(sum(s.startswith("torres")    for s in rad_src))),
        ("Mann 2015 ($T_\mathrm{eff}$)", int(sum(s.startswith("mann_teff") for s in rad_src))),
        ('ZAMS power law',     int(sum(s.startswith("ms(")       for s in rad_src))),
        ('Derived ($\log g$)', int(sum(s.startswith("logg_derived") for s in rad_src))),
        ('Unavailable',        int(sum(s == ""                   for s in rad_src))),
    ], False),
    (r'$M_\star$ source', [
        ('NEA',              int(sum(s == "nea"              for s in mass_src))),
        ('EU',               int(sum(s == "eu"                 for s in mass_src))),
        ('EPIC',             int(sum(s == "epic"             for s in mass_src))),
        ('Derived ($\log g$)', int(sum(s.startswith("logg_derived")   for s in mass_src))),
        ('Unavailable',      int(sum(s == ""                 for s in mass_src))),
    ], False),
    (r'$\log g$ source', [
        ('NEA',              int(sum(s == "nea"              for s in logg_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in logg_src_col))),
        ('TOI',              int(sum(s == "toi"              for s in logg_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in logg_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in logg_src_col))),
    ], False),
    (r'$[\mathrm{Fe/H}]$ source', [
        ('NEA',              int(sum(s == "nea"              for s in met_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in met_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in met_src_col))),
        ('TOI',              int(sum(s == "toi"              for s in met_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in met_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in met_src_col))),
    ], False),
    (r'$L_\star$ source', [
        ('NEA',              int(sum(s == "nea"              for s in lum_src))),
        ('EPIC',             int(sum(s == "epic"             for s in lum_src))),
        ('Computed',         int(sum(s.startswith("derived(") for s in lum_src))),
        ('Unavailable',      int(sum(s == ""                 for s in lum_src))),
    ], False),
    (r'Spectral type source', [
        ('NEA',              int(sum(s == "nea"              for s in spec_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in spec_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in spec_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in spec_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in spec_src_col))),
    ], False),
    (r'$V_\mathrm{mag}$ source', [
        ('NEA',              int(sum(s == "nea"              for s in vmag_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in vmag_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in vmag_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in vmag_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in vmag_src_col))),
    ], False),
    (r'$K_\mathrm{mag}$ source', [
        ('NEA',              int(sum(s == "nea"              for s in kmag_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in kmag_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in kmag_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in kmag_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in kmag_src_col))),
    ], False),
    (r'Distance $d$ source', [
        ('NEA',              int(sum(s == "nea"              for s in dist_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in dist_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in dist_src_col))),
        ('TOI',              int(sum(s == "toi"              for s in dist_src_col))),
        ('SIMBAD',           int(sum(s == "simbad"           for s in dist_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in dist_src_col))),
    ], False),
    (r'Period $P$ source', [
        ('Input',            int(sum(s == "input"            for s in p_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in p_src_col))),
    ], False),
    (r'Semi-major axis $a$ source', [
        ('Input',            int(sum(s == "input"            for s in a_src))),
        ('Kepler (P, M_star)',int(sum(s.startswith("kepler") for s in a_src))),
        ('Unavailable',      int(sum(s == ""                 for s in a_src))),
    ], False),
    (r'Planet radius $r_p$ source', [
        ('Transit direct',      int(np.sum(r_direct))),
        ('msini bounds (CK17)', int(np.sum(r_bounds))),
        ('Unavailable',         int(np.sum(r_none))),
    ], False),
    (r'Planet mass $m_p$ source', [
        ('NEA',              int(sum(s == "nea"              for s in pl_mass_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in pl_mass_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in pl_mass_src_col))),
    ], False),
    (r'Minimum mass $m \sin i$ source', [
        ('Input',            int(sum(s == "input"            for s in msini_src_col))),
        ('NEA',              int(sum(s == "nea"              for s in msini_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in msini_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in msini_src_col))),
    ], False),
    (r'Insolation flux $F_\mathrm{rel}$ source', [
        ('Direct (NEA insol)',  int(sum(s == "nea"       for s in flux_src))),
        ('Direct (EPIC insol)', int(sum(s == "epic"      for s in flux_src))),
        ('Direct (TOI insol)',  int(sum(s == "toi"       for s in flux_src))),
        ('Computed (L/a^2)',    int(sum(s.startswith("derived(")     for s in flux_src))),
        ('Unavailable',         int(sum(s == ""                for s in flux_src))),
    ], False),
    (r'Equilibrium temp $T_\mathrm{eq}$ source', [
        ('NEA',              int(sum(s == "nea"              for s in eqt_src_col))),
        ('EU',               int(sum(s == "eu"                 for s in eqt_src_col))),
        ('EPIC',             int(sum(s == "epic"             for s in eqt_src_col))),
        ('TOI',              int(sum(s == "toi"              for s in eqt_src_col))),
        ('Computed',         int(sum(s.startswith("derived(") for s in eqt_src_col))),
        ('Unavailable',      int(sum(s == ""                 for s in eqt_src_col))),
    ], False),
    ('Classification (overlapping)', [
        ('Confirmed rocky',              int(np.sum(rocky_conf))),
        ('Uncertain rocky (msini,CK17)', int(np.sum(rocky_unc))),
        ('Temperate HZ',                 int(np.sum(temperate))),
        ('Temperate + uncertain rocky',  int(np.sum(temperate & (rocky_conf | rocky_unc)))),
    ], True),
]

# ── Build and display diagram ─────────────────────────────────────────────
# Arrow convention (agreed rule):
#   solid  (--> )  = the value on this edge is DIRECTLY MEASURED from a catalog source
#   dashed (-.->) = the value on this edge is COMPUTED or ESTIMATED
# Inputs to computation nodes are solid (measured value flows in).
# Outputs from computation nodes are dashed (output is the derived quantity).
# Dual-arrow pattern at merge nodes: solid = measured rows, dashed = computed rows.

TEMPLATE = r'''```mermaid
%%{init: {"theme": "base", "layout": "elk", "elk": {"nodePlacementStrategy": "BRANDES_KOEPF"}}}%%
flowchart TB
    subgraph CAT["EMC Catalog"]
            direction LR
            R_J["$$r_p$$"]
            MSI["$$m\sin i$$"]
            A_C["$$a$$"]
            P_C["$$P$$"]
        end
    subgraph SRCS["Stellar Param Sources"]
        direction TB
        subgraph CATS["Input Catalogs"]
            direction TB
            NEA["<b>NASA Exoplanet Archive</b>"]
            EU["<b>Exoplanets Encyclopaedia</b>"]
            EPIC["<b>EPIC (K2)</b>"]
            TOI["<b>TOI (TESS)</b>"]
            SIM["<b>SIMBAD</b>"]
        end
        NEA  -->|"<<n_kmag_nea>>"| KMAG["K mag"]
        EPIC -->|"<<n_kmag_epic>>"| KMAG
        SIM  -->|"<<n_kmag_simbad>>"| KMAG
        EU   -->|"<<n_kmag_eu>>"| KMAG
        NEA  -->|"<<n_spec_nea>>"| SPEC["spec type"]
        EPIC -->|"<<n_spec_epic>>"| SPEC
        SIM  -->|"<<n_spec_simbad>>"| SPEC
        EU   -->|"<<n_spec_eu>>"| SPEC
        TEFF -->|"<<n_spec_zams>>"| SPEC
        NEA  -->|"<<n_teff_nea>>"| TEFF["$$T_{\mathrm{eff}}$$"]
        EPIC -->|"<<n_teff_epic>>"| TEFF
        TOI  -->|"<<n_teff_toi>>"| TEFF
        SIM  -->|"<<n_teff_simbad>>"| TEFF
        EU   -->|"<<n_teff_eu>>"| TEFF
        NEA  -->|"<<n_logg_nea>>"| LOGG["$$\log g$$"]
        EPIC -->|"<<n_logg_epic>>"| LOGG
        TOI  -->|"<<n_logg_toi>>"| LOGG
        SIM  -->|"<<n_logg_simbad>>"| LOGG
        NEA  -->|"<<n_met_nea>>"| MET["$$[\text{Fe/H}]$$"]
        EPIC -->|"<<n_met_epic>>"| MET
        TOI  -->|"<<n_met_toi>>"| MET
        SIM  -->|"<<n_met_simbad>>"| MET
        EU   -->|"<<n_met_eu>>"| MET
        NEA  -->|"<<n_rad_nea>>"| RSTAR["$$R_\star$$"]
        EPIC -->|"<<n_rad_epic>>"| RSTAR
        TOI  -->|"<<n_rad_toi>>"| RSTAR
        EU   -->|"<<n_rad_eu>>"| RSTAR
        TEFF --> |"<<n_rad_zams>>"| ZAMS["<b>[4] ZAMS Fallback</b><br/>$$R_\star\approx(T_{\mathrm{eff}}/T_\odot)^{n(spec)}$$"]
        SPEC -->|"<<n_zams_with_spec>>"| ZAMS
        ZAMS -. "<<n_rad_zams>>" .-> RSTAR
        KMAG -->|"<<n_rad_mann_mks>>"| MANN["<b>[1] Mann Mks</b><br/>$$R_\star=f(M_{K_s},[\text{Fe/H}])$$"]
        MET  -->|"<<n_met_mann_mks>>"| MANN
        MANN -. "<<n_rad_mann_mks>>" .-> RSTAR
        TEFF -->|"<<n_rad_mann_teff>>"| MANNTEFF["<b>[3] Mann Teff</b><br/>$$R_\star=f(T_{\text{eff}},[\text{Fe/H}])$$"]
        MET  -->|"<<n_met_mann_teff>>"| MANNTEFF
        MANNTEFF -. "<<n_rad_mann_teff>>" .-> RSTAR
        TEFF -->|"<<n_rad_torres>>"| TORRES["<b>[2] Torres 2010</b><br/>$$R_\star=f(T_{\text{eff}},\log g,[\text{Fe/H}])$$"]
        LOGG -->|"<<n_rad_torres>>"| TORRES
        MET  -->|"<<n_met_torres>>"| TORRES
        TORRES -. "<<n_rad_torres>>" .-> RSTAR
        NEA  -->|"<<n_mass_nea>>"| MSTAR["$$M_\star$$"]
        EPIC -->|"<<n_mass_epic>>"| MSTAR
        EU   -->|"<<n_mass_eu>>"| MSTAR
        LOGG  --> |"<<n_mass_logg>>"| MLOG["$$M_\star=10^{\log g-\log g_\odot+2\log_{10}R_\star}$$"]
        RSTAR --> |"<<n_mass_logg>>"| MLOG
        MLOG  -. "<<n_mass_logg>>" .-> MSTAR
        NEA   -->|"<<n_lum_nea>>"| LSTAR["$$L_\star$$"]
        EPIC  -->|"<<n_lum_epic>>"| LSTAR
        RSTAR -->|"<<n_lum_calc>>"| LUM["$$L_\star=R_\star^2(T_{\mathrm{eff}}/T_\odot)^4$$"]
        TEFF  --> |"<<n_lum_calc>>"| LUM
        LUM   -. "<<n_lum_calc>>" .-> LSTAR
    end

    FCALC["$$F=L_\star/a^2$$"]
    MSTAR -->|"<<n_a_kepler>>"| KEPL["$$a = \left(M_\star \left(\frac{P}{yr}\right)^2\right)^{1/3}$$"]
    P_C   -->|"<<n_p_valid>>"| KEPL

    A_C  -->|"<<n_a_direct>>"| AFIN["$$a$$"]
    KEPL -. "<<n_a_kepler>>" .-> AFIN
    AFIN  -->|"<<n_a_direct>>"| FCALC
    AFIN  -. "<<n_a_kepler>>" .-> FCALC
    LSTAR  -->|"<<n_lum_direct>>"| FCALC
    LSTAR  -. "<<n_lum_calc>>" .-> FCALC

    subgraph CLASS["Classification"]
        direction LR
        ROCKY[/"rocky$$\; r_p\in[0.5,1.5]\,R_\oplus$$"/]
        TEMP[/"temperate$$\; F_{\mathrm{rel}}\in[0.35,1.7]\,S_\oplus$$"/]
    end

    NEA   -->|"<<n_flux_nea>>"| FLUX["$$F_{\mathrm{rel}}$$"]
    EPIC  -->|"<<n_flux_epic>>"| FLUX
    TOI   -->|"<<n_flux_toi>>"| FLUX
    FCALC -. "<<n_flux_comp>>" .-> FLUX
    FLUX  -->|"<<n_flux_direct>>"| TEMP
    FLUX  -. "<<n_flux_comp>>" .-> TEMP

    R_J -->|"<<n_r_direct>>"| REARTH["$$r_p$$ (or bounds)"]
    MSI -->|"<<n_r_msini>>"| CK["M-R-rel + msini + i prior [1,2]"]
    CK  -. "<<n_r_msini>>" .-> REARTH
    REARTH  -->|"<<n_r_direct>>"| ROCKY
    REARTH  -. "<<n_r_msini>>" .-> ROCKY
    style NEA  fill:#dde8f7,stroke:#1a6fba,color:#1a6fba
    style EU   fill:#fff4d6,stroke:#b8860b,color:#8a6508
    style EPIC fill:#fde9dc,stroke:#d4550d,color:#d4550d
    style TOI  fill:#dcf5dc,stroke:#2a8a2a,color:#2a8a2a
    style SIM  fill:#f0e0fa,stroke:#7b42a8,color:#7b42a8
    <<LINKSTYLES>>
```'''

diagram = TEMPLATE
for key, val in counts.items():
    diagram = diagram.replace(f'<<{key}>>', val)

# drop EU edges that carry zero rows, then color source-origin edges by catalog.
# linkStyle indices refer to edges by order of appearance, so they are generated
# after pruning rather than hand-maintained.
_lines = [l for l in diagram.split('\n')
          if not (l.strip().startswith('EU') and '|"0"|' in l)]
_edge_idx, _i = {}, 0
for _l in _lines:
    if '-->' in _l or '.->' in _l:
        _edge_idx.setdefault(_l.strip().split()[0], []).append(_i)
        _i += 1
_LINK_COLORS = {'NEA': '#1a6fba', 'EU': '#b8860b', 'EPIC': '#d4550d', 'TOI': '#2a8a2a', 'SIM': '#7b42a8'}
_linkstyles = '\n'.join(f"    linkStyle {','.join(map(str, _edge_idx[n]))} stroke:{c},stroke-width:2px"
                         for n, c in _LINK_COLORS.items() if _edge_idx.get(n))
diagram = '\n'.join(_linkstyles if _l.strip() == '<<LINKSTYLES>>' else _l for _l in _lines)

print(diagram)
Markdown(diagram)


In [ ]:
for stage, cats, overlapping in branches:
    note = '  [counts may overlap]' if overlapping else ''
    print(f'\n── {stage}{note}')
    for label, count in cats:
        pct = 100 * count / n
        print(f'  {label:<35s}  {count:6,}  ({pct:5.1f}%)')


In [ ]:
from pathlib import Path

MMD_PATH = common.ROOT / "report_plots" / "enrichment_pipeline_flowchart.mmd"
MMD_PATH.write_text(diagram.strip("```").removeprefix("mermaid"), encoding="utf-8")


In [ ]:
%%bash
/home/joshuadreier/.local/share/nvm/v24.18.0/bin/mmdc -i report_plots/enrichment_pipeline_flowchart.mmd -o report/figures/enrichment_pipeline_flowchart.pdf
